# Rung 0 — recomputing every claim from the run's own artifacts

This notebook *proves*; `summary.ipynb` *explains*. Nothing here is imported from this project:
every number below is recomputed from a committed table using the Python standard library
(`csv`, `gzip`, `hashlib`, `statistics`) and arithmetic written out in full, so what you read is
exactly what is computed. A notebook that only called the verification script would relocate the
trust rather than discharge it; that script appears once, in the last cell, as a cross-check.

**What this rung measures.** For each (cell line, drug) condition the replicate plates are split
into two groups, each group's per-gene log2 fold change is averaged, and the two averaged
profiles are correlated across genes. That correlation is computed twice from the same split:
over **all** genes, and over the **responder** genes the condition's *first* group called
differentially expressed. Both are corrected to full length by Spearman-Brown, `2r / (1 + r)`,
and read against mismatched-condition floors. Every statistic below therefore comes in two
families, `all_*` and `responder_*`, in one summary row.

**What cannot be checked here, stated rather than hidden.** The 1,026 Tahoe pseudobulk shards
live on cluster scratch and are far too large to commit, so shard integrity reduces here to the
committed manifest's content hash. A promoted copy under `results/` does not exist until after
gate 2, and the permutation check is a separate cluster job; where those are absent the cell says
so and moves on, rather than failing on the calendar.

**Before the run.** The artifacts are uncommitted between the run and promotion (PROCESS, "What
reaches GitHub, and when"). Every cell degrades to "artifact not present yet" until they arrive,
so this notebook executes end to end on a fresh checkout.

In [ ]:
import csv
import gzip
import hashlib
import json
import math
import statistics
import subprocess
import sys
from pathlib import Path


def find_repo(start: Path) -> Path:
    """The repository root: the first ancestor carrying pyproject.toml."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return start


REPO = find_repo(Path.cwd().resolve())
TASK = "rung0-assay-reliability"
TASK_DIR = REPO / "docs" / "tasks" / TASK
TRANCHE = "tahoe100m-pseudobulk-de.v1"

#: gene set -> the per-condition table column holding that set's correlation
GENE_SETS = {"all": "r", "responder": "r_responder"}

FIGURES = [
    "01_build.png",
    "02_split.png",
    "03_select.png",
    "04_score.png",
    "05_decompose.png",
    "06_null.png",
    "07_terciles.png",
    "08_power.png",
    "09_per_gene_reliability.png",
]

print(f"repository  {REPO}")
print(f"task dir    {TASK_DIR}")
print(f"present     {(TASK_DIR / 'rung0_reliability.csv').exists()}")

### The helpers, written out once

Four small functions and nothing else: read a table (plain or gzipped) into a list of dictionaries,
turn a cell into a number, decide whether a reported value is a recomputed one rounded to its last
printed place, and print claim / recomputed / verdict. `read_rows` returns `None` when the run has
not happened yet, which is what lets every cell below degrade to a message instead of a traceback.

One detail that matters for correctness: one of the screen's fifty cell lines has a missing DepMap
identifier and appears throughout as the literal string `NA`. The `csv` module keeps it as text,
which is why the keys join to themselves here.

In [ ]:
def read_rows(name, directory=None):
    """Rows of a committed table as dictionaries, or None when it has not been written yet."""
    path = (directory or TASK_DIR) / name
    if not path.exists():
        print(f"artifact not present yet: {path}")
        return None
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", newline="") as handle:
        return list(csv.DictReader(handle))


def num(cell):
    """A CSV cell as a float; an empty cell is a missing value, written as nan."""
    text = (cell or "").strip()
    return float("nan") if text == "" else float(text)


def close(claim, recomputed, decimals):
    """True when the reported value is the recomputed one rounded to `decimals` places."""
    if math.isnan(claim) or math.isnan(recomputed):
        return math.isnan(claim) and math.isnan(recomputed)
    return abs(claim - recomputed) <= 0.5 * 10**-decimals + 1e-12


def verdict(name, claim, recomputed, ok):
    print(f"{name}\n  claim      : {claim}\n  recomputed : {recomputed}")
    print(f"  verdict    : {'PASS' if ok else 'FAIL'}\n")


def sha256_of(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


summary_rows = read_rows("rung0_reliability.csv")
summary = summary_rows[0] if summary_rows else None
per_pair = read_rows("rung0_per_pair_r.csv")

## Claim 1 — the reported condition count is the number of conditions actually scored

`n_pairs` is the number of conditions whose correlation is finite, not the number of rows in the
per-condition table. A condition that fell below the fifty-gene scoring threshold is kept in the
table, honestly blank, and is not one of the conditions the mean is taken over. The two families
differ here: a condition can be scoreable over all genes and unscoreable over its responders.

In [ ]:
if summary and per_pair:
    for label, column in GENE_SETS.items():
        values = [num(row[column]) for row in per_pair]
        finite = [v for v in values if not math.isnan(v)]
        verdict(
            f"{label}: n_pairs is the count of finite per-condition correlations",
            f"reported n_pairs {summary[f'{label}_n_pairs']}",
            f"{len(finite)} finite of {len(per_pair)} rows in rung0_per_pair_r.csv",
            len(finite) == int(summary[f"{label}_n_pairs"]),
        )

## Claim 2 — the mean, median and quartiles are the ones the per-condition values give

The declared statistic is the mean over conditions of the per-condition correlation, with the
median and quartiles beside it because a mean alone cannot say whether the reproducibility is
spread evenly or carried by a few conditions. All four are recomputed here from the same committed
values, using the standard library's quantiles (inclusive method, which is the interpolation the
run used) rather than any library the run itself called.

In [ ]:
if summary and per_pair:
    for label, column in GENE_SETS.items():
        finite = [num(row[column]) for row in per_pair]
        finite = [v for v in finite if not math.isnan(v)]
        q1, median, q3 = statistics.quantiles(finite, n=4, method="inclusive")
        for what, key, value in (
            ("mean", "splithalf_mean_r", statistics.fmean(finite)),
            ("median", "splithalf_median_r", median),
            ("lower quartile", "splithalf_q1_r", q1),
            ("upper quartile", "splithalf_q3_r", q3),
        ):
            verdict(
                f"{label}: {what} recomputes from the per-condition correlations",
                f"reported {key} {summary[f'{label}_{key}']}",
                f"over {len(finite)} committed values: {value:.4f}",
                close(num(summary[f"{label}_{key}"]), value, 3),
            )

## Claim 3 — Spearman-Brown is `2r / (1 + r)` of that mean, and again on the equal-halves subset

A split-half correlation is a reliability at half length. `2r / (1 + r)` corrects it to the full
screen's length, and the correction assumes the two halves are the same size. Three quarters of
this screen's conditions split one plate against two, so the corrected value is reported a second
time over the conditions with an even plate count — where the split is exact — and the gap between
the two is the size of that assumption rather than an argument about it.

The correction is applied to the mean over conditions, not per condition and then averaged: `2r /
(1 + r)` is not linear, so those two differ, and the design's declared statistic is the mean.

In [ ]:
if summary and per_pair:
    even_flag = [row["n_plates_even"].strip().lower() in ("true", "1") for row in per_pair]
    for label, column in GENE_SETS.items():
        values = [num(row[column]) for row in per_pair]
        finite = [v for v in values if not math.isnan(v)]
        even = [v for v, flag in zip(values, even_flag, strict=True) if flag and not math.isnan(v)]
        mean = statistics.fmean(finite)
        corrected = 2 * mean / (1 + mean)
        verdict(
            f"{label}: Spearman-Brown correction is 2r/(1+r) of that mean",
            f"reported spearman_brown_full {summary[f'{label}_spearman_brown_full']}",
            f"2 x {mean:.4f} / (1 + {mean:.4f}) = {corrected:.4f}",
            close(num(summary[f"{label}_spearman_brown_full"]), corrected, 3),
        )
        mean_even = statistics.fmean(even) if even else float("nan")
        corrected_even = 2 * mean_even / (1 + mean_even) if even else float("nan")
        verdict(
            f"{label}: the equal-halves subset is counted and corrected the same way",
            f"reported n_pairs_even {summary[f'{label}_n_pairs_even']}, mean "
            f"{summary[f'{label}_splithalf_mean_r_even_plates']}, corrected "
            f"{summary[f'{label}_spearman_brown_full_even_plates']}",
            f"{len(even)} equal-halves conditions, mean {mean_even:.4f}, "
            f"corrected {corrected_even:.4f}",
            len(even) == int(summary[f"{label}_n_pairs_even"])
            and close(num(summary[f"{label}_splithalf_mean_r_even_plates"]), mean_even, 3)
            and close(num(summary[f"{label}_spearman_brown_full_even_plates"]), corrected_even, 3),
        )

## Claim 4 — the positive fraction is the share of conditions above zero

`frac_pos` says how much of the reproducibility is general rather than carried by a handful of
conditions. It is a plain count over the same committed values.

In [ ]:
if summary and per_pair:
    for label, column in GENE_SETS.items():
        finite = [num(row[column]) for row in per_pair]
        finite = [v for v in finite if not math.isnan(v)]
        positive = sum(1 for v in finite if v > 0)
        verdict(
            f"{label}: frac_pos is the share of conditions above zero",
            f"reported frac_pos {summary[f'{label}_frac_pos']}",
            f"{positive} of {len(finite)} above zero: {positive / len(finite):.4f}",
            close(num(summary[f"{label}_frac_pos"]), positive / len(finite), 3),
        )

## Claim 5 — each chance floor is the mean of the mismatched-condition draws behind it

A floor is what a correlation of this kind is worth by construction alone. Three strata: *any
pair*; *different drug and line*, the generic-structure floor; and *same drug, different line*,
the stricter floor, since two lines given one drug share that drug's generic response. The summary
reports only each stratum's mean, so the run exported every individual draw and the means are
recomputed from them here. `null_n_draws` is the size of the stratum the p-value is read against.

In [ ]:
draws = read_rows("rung0_null_draws.csv")
if summary and draws:
    for label in GENE_SETS:
        subset = [row for row in draws if row["gene_set"] == label]
        counts = {}
        for stratum in ("any_pair", "diff_drug", "same_drug"):
            values = [num(row["r"]) for row in subset if row["stratum"] == stratum]
            counts[stratum] = len(values)
            mean = statistics.fmean(values) if values else float("nan")
            key = f"{label}_null_{stratum}_mean_r"
            verdict(
                f"{label}: the {stratum} floor recomputes from its draws",
                f"reported {key} {summary[key]}",
                f"mean of {len(values)} committed draws: {mean:.4f}",
                bool(values) and close(num(summary[key]), mean, 3),
            )
        expected = counts["diff_drug"] or counts["any_pair"]
        verdict(
            f"{label}: null_n_draws counts the stratum the p-value is read against",
            f"reported null_n_draws {summary[f'{label}_null_n_draws']}",
            f"{counts['diff_drug']} diff_drug draws (any_pair fallback: {counts['any_pair']})",
            expected == int(summary[f"{label}_null_n_draws"]),
        )

## Claim 6 — the observed mean clears both floors, and each comparison states its power

A reliability is only a reliability if it sits above what mismatched conditions give for free, so
the observed mean is required to exceed both floors it is read against. Beside it, each comparison
carries its minimum detectable effect at α = 0.05 and power 0.80 from the same bootstrap as its
p-value: a null result without its MDE cannot be told apart from an underpowered one, which is the
distinction the later rungs' small cohorts turn on. Here the MDEs are required to exist — positive
and finite — because a comparison with no computable detection threshold reports no power at all.

In [ ]:
if summary:
    for label in GENE_SETS:
        mean = num(summary[f"{label}_splithalf_mean_r"])
        floor_diff = num(summary[f"{label}_null_diff_drug_mean_r"])
        floor_same = num(summary[f"{label}_null_same_drug_mean_r"])
        verdict(
            f"{label}: the observed mean clears both floors",
            "mean > different-drug floor and > same-drug floor",
            f"{mean} > {floor_diff} and {mean} > {floor_same}",
            mean > floor_diff and mean > floor_same,
        )
        mdes = [
            num(summary[f"{label}_mde_80_vs_diff_drug"]),
            num(summary[f"{label}_mde_80_vs_same_drug"]),
        ]
        verdict(
            f"{label}: both minimum detectable effects are positive and finite",
            "an MDE at alpha 0.05, power 0.80 against each floor",
            f"vs different-drug {mdes[0]}, vs same-drug {mdes[1]}",
            all(math.isfinite(m) and m > 0 for m in mdes),
        )

## Claim 7 — reproducibility rises with effect size (the empirical in-run control)

Conditions are cut into thirds by response size and the split-half mean must rise across the
thirds. An assay that cannot find more reproducibility where there is more signal is broken, so a
failure here is a finding about the screen rather than a bug in this notebook — which is why the
three means are printed whatever the verdict.

In [ ]:
terciles = read_rows("rung0_effect_terciles.csv")
if terciles:
    means = [num(row["mean_r"]) for row in sorted(terciles, key=lambda r: int(r["tercile"]))]
    rises = len(means) == 3 and means[0] < means[1] < means[2]
    verdict(
        "reproducibility rises with effect size",
        "tercile 1 < tercile 2 < tercile 3 of the split-half mean",
        " -> ".join(f"{m:.4f}" for m in means),
        rises,
    )

## Claim 8 — selecting responders from both halves inflates a correlation out of nothing

The responder genes are chosen from the first plate group alone. Choosing them from the two halves
pooled would inflate the correlation by winner's curse: writing the halves as *a* and *b*, their
sum and difference are independent, so selecting on a large `|a + b|` inflates `var(a + b)` alone
and `cov(a, b) = (var(a+b) - var(a-b)) / 4` goes positive with nothing generating it. The run
measures both rules on a pool with no signal at all, and the pooled rule must come back visibly
higher. That gap is the size of the error the one-sided rule avoids.

In [ ]:
leakage = read_rows("rung0_leakage_control.csv")
if leakage:
    by_rule = {row["rule"]: num(row["mean_r"]) for row in leakage}
    one, pooled = by_rule.get("one-sided", float("nan")), by_rule.get("pooled", float("nan"))
    verdict(
        "two-sided selection inflates a signal-free correlation, one-sided does not",
        "pooled mean r > one-sided mean r on a pool with no signal",
        f"pooled {pooled} vs one-sided {one}",
        math.isfinite(one) and math.isfinite(pooled) and pooled > one,
    )

## Claim 9 — the noise decomposition, from the per-gene rows and by its own identity

`lfcSE` is the standard error of one plate's treated-versus-control contrast: cell-sampling error
at that row's cell counts. It cannot see plate-to-plate variation — culture day, handling,
position. Across plates at a fixed dose the fold change varies by both, so

    sigma2_plate = var_across_plates(log2FoldChange) - mean(lfcSE^2), floored at zero

estimates the plate component alone, and `between_plate_fraction` is its share of the delta's
replicate variance. Two things are checked: the reported fraction is the mean of the per-gene
fractions, and the identity above holds row by row. The identity is what makes this a
decomposition rather than a ratio of two stored numbers.

This cell streams the per-gene table one row at a time — the mean is exact over every row; the
identity is checked on the first 200,000, which settles an algebraic identity that holds row-wise
or not at all. On the full screen the pass takes a minute or two.

In [ ]:
IDENTITY_SAMPLE = 200_000

noise_summary = read_rows("rung0_noise_decomposition.csv")
noise_path = TASK_DIR / "rung0_noise_per_gene.csv.gz"
if noise_summary and noise_path.exists():
    reported = noise_summary[0]
    n_rows = total = n_finite = worst = 0
    total = 0.0
    worst = 0.0
    with gzip.open(noise_path, "rt", newline="") as handle:
        for row in csv.DictReader(handle):
            n_rows += 1
            fraction = num(row["between_plate_fraction"])
            if math.isfinite(fraction):
                total += fraction
                n_finite += 1
            if n_rows <= IDENTITY_SAMPLE:
                expected = max(num(row["var_lfc"]) - num(row["mean_se2"]), 0.0)
                worst = max(worst, abs(expected - num(row["sigma2_plate"])))
    mean_fraction = total / n_finite if n_finite else float("nan")
    verdict(
        "the per-gene noise table has the row count the summary reports",
        f"reported n_gene_conditions {reported['n_gene_conditions']}",
        f"{n_rows} rows in rung0_noise_per_gene.csv.gz",
        n_rows == int(num(reported["n_gene_conditions"])),
    )
    verdict(
        "the between-plate fraction is the mean of the per-gene fractions",
        f"reported between_plate_fraction_mean {reported['between_plate_fraction_mean']}",
        f"mean of {n_finite} finite per-gene fractions: {mean_fraction:.6f}",
        close(num(reported["between_plate_fraction_mean"]), mean_fraction, 4),
    )
    verdict(
        "sigma2_plate = max(var_lfc - mean_se2, 0), row by row",
        f"the identity holds on each of the first {min(n_rows, IDENTITY_SAMPLE):,} rows",
        f"worst absolute deviation {worst:.3e}",
        worst < 1e-9,
    )
elif noise_summary:
    print(f"artifact not present yet: {noise_path}")

## Claim 10 — every example scatter reproduces the correlation it is shown under

Every correlation in this analysis is one point in a distribution, so the run exports both halves'
per-gene values for a few example conditions spanning the reliability range, plus the two
mismatched comparisons the floors are built from. Recomputing each correlation from its own
exported points is what stops a caption asserting a number the plotted data does not support.

In [ ]:
profiles = read_rows("rung0_example_pair_profiles.csv.gz")
index = read_rows("rung0_example_pair_index.csv")
if profiles and index:
    points = {}
    for row in profiles:
        points.setdefault(row["example_id"], []).append((num(row["lfc0"]), num(row["lfc1"])))
    for entry in index:
        pairs = points.get(entry["example_id"], [])
        xs = [x for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        ys = [y for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        recomputed = statistics.correlation(xs, ys) if len(xs) > 1 else float("nan")
        verdict(
            f"example {entry['example_id']} ({entry['kind']}) reproduces its correlation",
            f"index r_shown {entry['r_shown']} over {entry['n_genes_shown']} genes",
            f"from the {len(pairs)} committed points: {recomputed:.4f}",
            close(num(entry["r_shown"]), recomputed, 4)
            and len(pairs) == int(entry["n_genes_shown"]),
        )

## Claim 11 — every declared figure exists, and the score figure's printed number reproduces

`design.md` names the figures before the run, so a reviewer sees the evidence the design promised
rather than the subset that looked best afterwards. Each is required to be a real PNG, not an empty
placeholder. The score figure additionally writes the points it drew and the correlation it printed
on each panel to a companion table, so the number on the image is recomputable from the image's own
data — a figure whose values live only inside a run cannot be checked at all.

In [ ]:
figure_dir = TASK_DIR / "figures"
if figure_dir.exists():
    present = [name for name in FIGURES if (figure_dir / name).exists()]
    real = [
        name
        for name in present
        if (figure_dir / name).stat().st_size > 5_000
        and (figure_dir / name).read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"
    ]
    verdict(
        "every figure design.md declares was written, and is a real image",
        f"{len(FIGURES)} figures, each a PNG over 5 kB",
        f"{len(present)} present, {len(real)} non-trivial",
        len(real) == len(FIGURES),
    )
else:
    print(f"artifact not present yet: {figure_dir}")

values = read_rows("04_score.values.csv", directory=figure_dir)
if values:
    panels = {}
    printed = {}
    for row in values:
        panels.setdefault(row["example_id"], []).append((num(row["lfc0"]), num(row["lfc1"])))
        printed[row["example_id"]] = num(row["r_printed"])
    for panel, pairs in panels.items():
        xs = [x for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        ys = [y for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        recomputed = statistics.correlation(xs, ys) if len(xs) > 1 else float("nan")
        verdict(
            f"score figure panel {panel} prints the correlation of its own points",
            f"printed r {printed[panel]:.4f}",
            f"from the {len(pairs)} plotted points: {recomputed:.4f}",
            close(printed[panel], recomputed, 4),
        )

## Claim 12 — the scored conditions are the splittable ones, and the two tables agree on which

`rung0_pool_description.csv` is the screen as it arrived: one row per candidate (line, drug) with
its plate count per half, measured rather than asserted. A condition reaches the per-condition
table exactly when both halves are non-empty, so the counts must agree by subtraction. The
equal-halves flag is recorded in both tables and defines the Spearman-Brown subset above, so it is
required to be the same flag in both places.

In [ ]:
pool = read_rows("rung0_pool_description.csv")
if pool and per_pair:
    unsplittable = [
        row for row in pool if int(row["n_plates_half0"]) == 0 or int(row["n_plates_half1"]) == 0
    ]
    verdict(
        "the scored conditions are the splittable ones, by subtraction",
        f"{len(pool)} candidate conditions - {len(unsplittable)} with an empty half",
        f"{len(pool) - len(unsplittable)} splittable; {len(per_pair)} per-condition rows",
        len(pool) - len(unsplittable) == len(per_pair),
    )
    pool_even = {
        (row["patient"], row["drug"]): row["n_plates_even"].strip().lower() for row in pool
    }
    agree = sum(
        1
        for row in per_pair
        if pool_even.get((row["patient"], row["drug"])) == row["n_plates_even"].strip().lower()
    )
    verdict(
        "the equal-halves flag agrees between the pool and the per-condition table",
        "n_plates_even is one flag, recorded twice",
        f"{agree} of {len(per_pair)} conditions agree",
        agree == len(per_pair),
    )

## Claim 13 — every recorded checksum recomputes from the file it names

The most important cell in this notebook. The audit reads these artifacts in the working tree,
before they are committed (PROCESS, "What reaches GitHub, and when"), so nothing else ties what a
reader audited to what a reviewer later pulls. `audit_checksums.json` records the sha256 of every
table and figure the run wrote; recomputing them now is what closes that window. A single altered
byte anywhere moves a hash and fails here.

In [ ]:
checksums_path = TASK_DIR / "audit_checksums.json"
if checksums_path.exists():
    recorded = json.loads(checksums_path.read_text())
    by_name = {path.name: path for path in TASK_DIR.rglob("*") if path.is_file()}
    missing = sorted(name for name in recorded if name not in by_name)
    moved = sorted(
        name
        for name, digest in recorded.items()
        if name in by_name and sha256_of(by_name[name]) != digest
    )
    detail = f"{len(recorded) - len(missing) - len(moved)} of {len(recorded)} match"
    verdict(
        "every recorded artifact checksum recomputes from the file it names",
        f"{len(recorded)} sha256 entries in audit_checksums.json",
        (
            detail
            + (f"; missing {missing}" if missing else "")
            + (f"; CHANGED {moved}" if moved else ""),
        ),
        not missing and not moved and bool(recorded),
    )
else:
    print(f"artifact not present yet: {checksums_path}")

## Claim 14 — the data pin: the tranche's content hash, recomputed from the committed manifest

The 1,026 shards are on cluster scratch and cannot be rehashed on a laptop. What can be rehashed is
the manifest committed beside the tranche record: `scripts/register_tranche.py` builds one
`relative path <tab> size <tab> sha256` line per shard and takes the sha256 of that text. Rebuilding
the text from the manifest and hashing it confirms the record and the manifest describe the same
1,026 files — the pin on which bytes the run read, without holding those bytes.

In [ ]:
record_path = REPO / "data" / "tranches" / f"{TRANCHE}.json"
manifest_path = REPO / "data" / "tranches" / f"{TRANCHE}.manifest.txt"
if record_path.exists() and manifest_path.exists():
    record = json.loads(record_path.read_text())
    lines = [line.split("\t") for line in manifest_path.read_text().splitlines()]
    text = "".join(f"{rel}\t{size}\t{sha}\n" for rel, size, sha in lines)
    recomputed = hashlib.sha256(text.encode()).hexdigest()
    verdict(
        "the tranche content hash recomputes from the committed manifest",
        f"record content_hash {record['content_hash']}",
        f"sha256 of the rebuilt manifest text {recomputed}",
        recomputed == record["content_hash"],
    )
    verdict(
        "the manifest describes the whole download",
        "1,026 shards (docs/DATA.md)",
        f"{len(lines)} manifest lines, each path/size/sha256",
        len(lines) == 1026 and all(len(line) == 3 for line in lines),
    )
else:
    print(f"artifact not present yet: {record_path}")

## Claim 15 — the permutation check, when it has run

Mismatched draws reuse the same half-profiles, so they are not independent and the bootstrap's
p-value could be optimistic. Permuting the pairing measures that dependence directly: each
permutation gives one null mean, and the exact p-value is `(1 + #{permutations at least as large as
the observed}) / (1 + #permutations)`. It is a separate cluster job — when its output is absent this
cell says so and moves on.

In [ ]:
for label, suffix in (("all", ""), ("responder", "_responder")):
    perm_summary = read_rows(f"rung0_permutation_summary{suffix}.csv")
    perm_means = read_rows(f"rung0_permutation_perm_means{suffix}.csv")
    if not (perm_summary and perm_means):
        continue
    row = perm_summary[0]
    means = [num(entry["perm_mean"]) for entry in perm_means]
    observed = num(row["observed_mean"])
    at_least = sum(1 for m in means if m >= observed)
    p_exact = (1 + at_least) / (1 + len(means))
    verdict(
        f"{label}: the permutation-mean summary recomputes from the draws",
        f"reported mean {row['perm_mean_mean']}, sd {row['perm_mean_sd']} over {row['n_perm']}",
        f"mean {statistics.fmean(means):.4f}, sd {statistics.stdev(means):.4f} over {len(means)}",
        len(means) == int(row["n_perm"])
        and close(num(row["perm_mean_mean"]), statistics.fmean(means), 4)
        and close(num(row["perm_mean_sd"]), statistics.stdev(means), 4),
    )
    verdict(
        f"{label}: the exact permutation p-value recomputes",
        f"reported p_exact {row['p_exact']}",
        f"(1 + {at_least}) / (1 + {len(means)}) = {p_exact:.4f}",
        close(num(row["p_exact"]), p_exact, 4),
    )
    verdict(
        f"{label}: the observed mean sits above every permutation",
        f"observed {observed} above the largest of {len(means)} permutation means",
        f"largest permutation mean {max(means):.4f}",
        observed > max(means),
    )

## Claim 16 — the promoted copies, once they exist

Promotion follows gate 2, so before it there is nothing under `results/` to disagree with and this
cell says so. Afterwards, each promoted table must be byte-identical to the task-side copy the
audit read, and the provenance record's `result_sha256` must recompute from the promoted file.

In [ ]:
promoted_dir = REPO / "results" / TASK
records = sorted(promoted_dir.glob("*.provenance.json")) if promoted_dir.is_dir() else []
if not records:
    print(f"not promoted yet (promotion follows gate 2): {promoted_dir}")
for provenance in records:
    record = json.loads(provenance.read_text())
    promoted = REPO / record["result"]
    task_side = TASK_DIR / promoted.name
    identical = task_side.exists() and task_side.read_bytes() == promoted.read_bytes()
    verdict(
        f"{promoted.name}: the promoted copy is the table the audit read",
        "byte-identical to the task-side copy, and its recorded sha256 recomputes",
        f"identical {identical}; sha256 matches {sha256_of(promoted) == record['result_sha256']}",
        identical and sha256_of(promoted) == record["result_sha256"],
    )

## Cross-check — the same claims through `scripts/verify_rung0.py`

Everything above was recomputed here, in the open. This last cell runs the project's verification
battery over the same artifacts, so the two paths are compared rather than one standing in for the
other. It is the notebook's cross-check, never its body: a notebook that only called this script
would relocate the trust instead of discharging it. The script exits 2 when the run has not
happened yet, which is what the message below reports on a fresh checkout.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(REPO / "scripts" / "verify_rung0.py"), "--task-dir", str(TASK_DIR)],
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout or completed.stderr)
print(f"exit status {completed.returncode}")